# RFQ Prioritisation

This notebook applies the scoring model to the synthetic RFQ queue and ranks requests by priority.

## Load the scoring model

The RFQ scoring logic is kept separately in `src/scoring.py`.

This keeps the notebook focused on the analysis and results, while the scoring logic can be reused independently and later connected to other data sources.

In [1]:
import sys
from pathlib import Path

# Add the project root to Python's search path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.scoring import score_rfqs

In [2]:
import pandas as pd

df = pd.read_csv("../data/synthetic_rfqs.csv")

df.head()

,client,product_type,underlying,direction,notional,underlying_move_pct,iv_change,distance_to_barrier_pct,bid_ask_spread_pct,time_since_last_price_update_min,underlying_move_5m_pct,iv_change_5m
0,Client_A,Autocall,NVDA,SELL,1500000,-7.0,10.0,4.0,1.5,25,-2.20,3.0
1,Client_B,Tracker,SMI,BUY,30000,0.3,0.5,NaN,0.2,2,0.05,0.1
2,Client_C,Barrier Reverse Convertible,AAPL,SELL,800000,-2.5,3.0,12.0,0.7,8,-0.40,0.5
3,Client_A,Autocall,TSLA,BUY,250000,4.5,6.0,7.0,1.0,15,1.10,1.5
4,Client_D,Bonus Certificate,NESN,SELL,1000000,-0.8,1.0,18.0,0.5,5,-0.10,0.2


In [3]:
scored_rfqs = score_rfqs(df)

## RFQ Priority Ranking

The final priority score combines barrier proximity, market movement, implied volatility changes, quote staleness and RFQ size.

Higher scores indicate RFQs that may require more immediate trader attention.

In [4]:
scored_rfqs[[
    "client",
    "product_type",
    "underlying",
    "direction",
    "notional",
    "priority",
    "priority_score",
    "priority_reasons"
]].sort_values(
    "priority_score",
    ascending=False
).round({"priority_score": 2})

,client,product_type,underlying,direction,notional,priority,priority_score,priority_reasons
0,Client_A,Autocall,NVDA,SELL,1500000,HIGH,0.92,Near barrier | Large market move | IV shock | ...
7,Client_F,Autocall,AMZN,BUY,2000000,HIGH,0.81,Near barrier | Large market move | IV shock | ...
3,Client_A,Autocall,TSLA,BUY,250000,MEDIUM,0.68,Near barrier | Large market move | IV shock | ...
2,Client_C,Barrier Reverse Convertible,AAPL,SELL,800000,MEDIUM,0.46,Near barrier
4,Client_D,Bonus Certificate,NESN,SELL,1000000,LOW,0.28,No major risk flag
6,Client_B,Reverse Convertible,MSFT,SELL,600000,LOW,0.16,No major risk flag
5,Client_E,Tracker,SPX,BUY,100000,LOW,0.03,No major risk flag
1,Client_B,Tracker,SMI,BUY,30000,LOW,0.03,No major risk flag
